In [1]:
import numpy as np
from netCDF4 import Dataset
import matplotlib.pyplot as plt
#import cartopy.crs as crs
from cartopy.io.shapereader import Reader
from cartopy.feature import ShapelyFeature
from cartopy.feature import NaturalEarthFeature
import cartopy.crs as ccrs
from datetime import datetime, timedelta
from metpy.units import units
from metpy.calc import dewpoint_from_specific_humidity, relative_humidity_from_specific_humidity, wind_speed
import pyart
import geopy.distance as distance
import haversine
from wrf import getvar


## You are using the Python ARM Radar Toolkit (Py-ART), an open source
## library for working with weather radar data. Py-ART is partly
## supported by the U.S. Department of Energy as part of the Atmospheric
## Radiation Measurement (ARM) Climate Research Facility, an Office of
## Science user facility.
##
## If you use this software to prepare a publication, please cite:
##
##     JJ Helmus and SM Collis, JORS 2016, doi: 10.5334/jors.119



In [2]:
#print(P2)

In [3]:
ncfile1 = Dataset('/glade/work/mawilson/DART_mpd/observations/utilities/threed_sphere/obs_epoch_100mIOP4.nc')
#ncfile1 = Dataset('/glade/work/mawilson/DART_mpd/observations/utilities/threed_sphere/obs_epoch_100mnew.nc')

In [4]:
location = ncfile1.variables['location'][:]
qc = ncfile1.variables['qc'][:]
obstype = ncfile1.variables['obs_type'][:]
obstypemd = ncfile1.variables['ObsTypesMetaData'][:]
obs_val = ncfile1.variables['observations'][:]
which_vert = ncfile1.variables['which_vert'][:]
times = ncfile1.variables['time'][:]
print(obstype[obstype==28])
print(len(obstype[obstype==28]))
qc_new = []
for i in range(len(qc)):
    qc_d = qc[i][0]
    qc_new.append(qc_d)
qc_new = np.asarray(qc_new)

[28 28 28 ... 28 28 28]
3936


In [5]:
print(np.unique(obstype))

[ 16  17  18  25  26  27  28 105 106 107 108 141 142]


In [6]:
otype_s = []
obs_s = []
lon_s = []
lat_s = []
elev_s = []
error_s = []
time_s = []

#Manually set errors for different otypes
# 1.2 K T err recommended by Soyoung Ha, use same for Td initially
err_T = 1.0
err_Td = 1.0
#Try a 2 m/s wind error
err_U = 1.5
err_V = 1.5

#minute_range = np.arange(180,185,5)
minute_range = np.arange(180,600,60)
#minute_range = np.arange(595,605,5)

for mins in minute_range:
    #dt_start = datetime(2022,9,15,0,0)
    dt_start = datetime(2022,7,19,0,0)
    #dt_start = datetime(2021,6,4,0,0)
    
    dt = dt_start + timedelta(minutes=int(mins))
    print(dt)
    #dt = datetime(2022,9,15,9,55)
    #wrfout = Dataset('/glade/derecho/scratch/mawilson/20210604_newcase/nature_100IOP6/final_nature/wrfout_d02_2022-'+str(dt.isoformat()[5:7])+'-'+str(dt.isoformat()[8:10])+'_'+str(dt.isoformat()[11:13])+':'+str(dt.isoformat()[14:16])+':00')
    #wrfout = Dataset('/glade/derecho/scratch/mawilson/20210604_newcase/nature_100IOP4/final_nature/wrfout_d02_2022-'+str(dt.isoformat()[5:7])+'-'+str(dt.isoformat()[8:10])+'_'+str(dt.isoformat()[11:13])+':'+str(dt.isoformat()[14:16])+':00')
    #wrfout = Dataset('/glade/campaign/ral/aap/mawilson/nature_runs/JUNE/final_nature/wrfout_d02_2021-'+str(dt.isoformat()[5:7])+'-'+str(dt.isoformat()[8:10])+'_'+str(dt.isoformat()[11:13])+':'+str(dt.isoformat()[14:16])+':00')
    wrfout = Dataset('/glade/campaign/ral/aap/mawilson/nature_runs/IOP4/final_nature/wrfout_d02_2022-'+str(dt.isoformat()[5:7])+'-'+str(dt.isoformat()[8:10])+'_'+str(dt.isoformat()[11:13])+':'+str(dt.isoformat()[14:16])+':00')
    
    lon = wrfout.variables['XLONG']
    lat = wrfout.variables['XLAT']
    U10 = wrfout.variables['U10']
    V10 = wrfout.variables['U10']
    T2 = np.asarray(wrfout.variables['T2'])*units('K')
    T2F = T2 .to('degF')
    Q2 = np.asarray(wrfout.variables['Q2'])
    P2 = np.asarray(wrfout.variables['PSFC'][:]/100) * units('hPa')
    Td2 = dewpoint_from_specific_humidity(P2[0,:,:], T2[0,:,:], Q2[0,:,:]*units('kg/kg'))
    RH2 = relative_humidity_from_specific_humidity(P2[0,:,:], T2[0,:,:], Q2[0,:,:]*units('kg/kg'))
    SPD10 = wind_speed(np.asarray(U10)*units('m/s'), np.asarray(V10)*units('m/s'))
    cloud=wrfout.variables['QCLOUD']
    z_zm = np.asarray(getvar(wrfout, "height"))
    
    
    #otype = 107
    #otype = 105
    #otype = 106
    #otype = 108
    #otype = 142
    otype_list = [25,26,27,28]
    for otype in otype_list:
        loc_T2 = location[obstype==otype, :]
        qc_T2 = qc_new[obstype==otype]
        obs_T2 = obs_val[obstype==otype, :]
        lons_T2 = loc_T2[:,0]
        lats_T2 = loc_T2[:,1]
        elev_T2 = loc_T2[:,2]
        time_T2 = times[obstype==otype]
        lons_T2[lons_T2 > 180] = lons_T2[lons_T2 > 180] - 360
        
        #Convert WRF file time into same units as the obs_seq time
        dt_tot = (dt - datetime(1601,1,1)).total_seconds() / 86400
        time_diff = np.abs(dt_tot - time_T2)
        
        #Get obs within +- 2.5 minutes of each WRF file
        time_T3 = time_T2[time_diff<(150/86400)]
        loc_T3 = loc_T2[time_diff<(150/86400), :]
        qc_T3 = qc_T2[time_diff<(150/86400)]
        lons_T3 = lons_T2[time_diff<(150/86400)]
        lats_T3 = lats_T2[time_diff<(150/86400)]
        elev_T3 = elev_T2[time_diff<(150/86400)]
        obs_T3 = obs_T2[time_diff<(150/86400), :]
        
        if len(obs_T3)==0:
            print('NO OBS IN WINDOW')
        for k in range(len(lons_T3)):
            latp=lats_T3[k]
            lonp=lons_T3[k]
            #Get location for each ob in model land
            lon1d = np.ndarray.flatten(lon[0,:,:])
            lat1d = np.ndarray.flatten(lat[0,:,:])
            station = []
            points = []
            for i in range(len(lon1d)):
                points.append((lat1d[i],lon1d[i]))
                station.append((latp,lonp))
            dist = haversine.haversine_vector(station,points)
            dist2=dist.reshape(lon.shape[1],lon.shape[2])
            print(lon[0,:,:][np.where(dist2==np.min(dist2))])
            print(lat[0,:,:][np.where(dist2==np.min(dist2))])
            print(np.where(dist2==np.min(dist2)), dt)
            st_xind = np.where(dist2==np.min(dist2))[0][0]
            st_yind = np.where(dist2==np.min(dist2))[1][0]
            #Get elevation from WRF, not bad obs_seq info
            elev_good = z_zm[0,st_xind,st_yind]
            if otype == 27:
                T2_a = T2[0,st_xind,st_yind].magnitude
                err_fin = err_T
            elif otype == 25:
                T2_a = U10[0,st_xind,st_yind]
                err_fin = err_U
            elif otype == 26:
                T2_a = V10[0,st_xind,st_yind]
                err_fin = err_V
            elif otype == 28:
                T2_a = (Td2[st_xind,st_yind].to('K')).magnitude
                err_fin = err_Td
                otype=127
            #If you want to change the error assumption, just change the scale in this line
            error = np.random.normal(loc=0.0, scale=np.sqrt(obs_T3[k,1]))
            #6/17/2024 MW adding code to limit added error to 1.5 times the error assumtion in DART
            if np.abs(error/4) > (np.sqrt(obs_T3[k,1])*1.0):
                error = (error / np.abs(error)) * (np.sqrt(obs_T3[k,1])*1.0)
            T2_b = T2_a + (error/4)
            print(T2_a, (error/4))
            print(elev_good)
        
            #Append obs to arrays for writing to obs_seq file later
            otype_s.append(otype)
            obs_s.append(T2_b)
            lon_s.append(lonp)
            lat_s.append(latp)
            elev_s.append(elev_good)
            error_s.append(err_fin**2)
            time_s.append(time_T3[k])

2022-07-19 03:00:00
[-84.51233]
[39.4292]
(array([760]), array([711])) 2022-07-19 03:00:00
-0.2914557 0.658317676039219
204.20782
[-84.88624]
[39.141964]
(array([471]), array([423])) 2022-07-19 03:00:00
1.2935028 -1.2605624903108854
238.48517
[-85.13965]
[39.641056]
(array([970]), array([226])) 2022-07-19 03:00:00
2.1755865 0.11689920963904467
274.22342
[-85.24827]
[39.072422]
(array([401]), array([142])) 2022-07-19 03:00:00
1.9498363 -0.24713988219831687
292.48236
[-84.79973]
[39.399544]
(array([729]), array([489])) 2022-07-19 03:00:00
0.45419326 -0.5747364038581579
291.31332
[-84.38455]
[39.289467]
(array([621]), array([811])) 2022-07-19 03:00:00
0.34707886 -1.741210222758156
258.0333
[-84.11635]
[39.348274]
(array([682]), array([1018])) 2022-07-19 03:00:00
1.499583 1.14383217807875
236.67952
[-84.6622]
[38.948265]
(array([278]), array([598])) 2022-07-19 03:00:00
1.1918961 0.8317588700658708
288.69077
[-84.41576]
[39.15376]
(array([485]), array([788])) 2022-07-19 03:00:00
0.67041487 

In [7]:
#Convert lats and lons to radians for DART, because why not
lon_DART = np.radians(np.asarray(lon_s))
lat_DART = np.radians(np.asarray(lat_s))

lon_DART = np.where(lon_DART > 0.0, lon_DART, lon_DART+(2.0*np.pi))

#Convert time into DART format. This is hacky now, improve later
#day_DART = 154024
day_DART = 153966
#day_DART = 153556

seconds_DART = (np.asarray(time_s) - day_DART) * 86400

In [8]:
#Sort everything in time order
inds_time = seconds_DART.argsort()
# print(seconds_DART)
# print(seconds_DART[inds_time])
seconds_DART1 = seconds_DART[inds_time]
seconds_DART1[seconds_DART1 < 0] = 0
obs_s1 = np.asarray(obs_s)[inds_time]
lon_DART1 = lon_DART[inds_time]
lat_DART1 = lat_DART[inds_time]
elev_s1 = np.asarray(elev_s)[inds_time]
otype_s1 = np.asarray(otype_s)[inds_time]
error_s1 = np.asarray(error_s)[inds_time]

In [9]:
for bigfoot in [1,2]:
    print(bigfoot)

    #Write the simulated obs out to an obs_seq file
    #filename = 'SIM_MESONET_IOP6_oneob'
    #filename = 'SIM_MESONET_IOP4_halferr'
    filename = 'SIM_MESONET_IOP4_z'
    
    fi = open(filename, "w")
    fi.write(" obs_sequence\n")
    fi.write("obs_kind_definitions\n")
    fi.write("    %d \n" %(4))
    fi.write("    %d          %s   \n" %(27, 'LAND_SFC_TEMPERATURE'))
    fi.write("    %d          %s   \n" %(25, 'LAND_SFC_U_WIND_COMPONENT'))
    fi.write("    %d          %s   \n" %(26, 'LAND_SFC_V_WIND_COMPONENT'))
    fi.write("    %d          %s   \n" %(127, 'LAND_SFC_DEWPOINT'))
    
    fi.write("  num_copies:            %d  num_qc:            %d\n" % (1,1))
    fi.write(" num_obs:       %d  max_num_obs:       %d\n" % (len(obs_s1), len(obs_s1)))
    fi.write("MADIS observation\n")
    fi.write("Data QC\n")
    
    fi.write("  first:            %d  last:       %d\n" % (1, len(obs_s1)))
    
    for q in range(len(obs_s1)):
    
        fi.write(" OBS            %d\n" % (q+1) )
        fi.write("   %20.14f\n" % obs_s1[q] )
        fi.write("   %20.14f\n" % 1.0 )
    
        if q+1 == 1:
            fi.write(" %d %d %d\n" % (-1, q+2, -1) ) #First obs
        elif q+1 == len(obs_s1):
            fi.write(" %d %d %d\n" % (q, -1, -1) ) #Last obs
        else:
            fi.write(" %d %d %d\n" % (q, q+2, -1) )
    
        fi.write("obdef\n")
        fi.write("loc3d\n")
        fi.write("    %20.14f          %20.14f          %20.14f     %d\n" %
                           (lon_DART1[q], lat_DART1[q], elev_s1[q], -1))
        fi.write("kind\n")
    
        fi.write("     %d     \n" % otype_s1[q])
    
        fi.write("    %d          %d     \n" % (seconds_DART1[q], day_DART))
    
        fi.write("    %20.14f  \n" % error_s1[q])

1
2


In [10]:
print(obs_T3[:,1])

#Get a randomly-generated error value to add to the synthetic ob 
#by sampling a Gaussian distribution with the same standard deviation as
#the assumed error from DART
error = np.random.normal(loc=0.0, scale=np.sqrt(obs_T3[0,1]))
print(error)

[8.54565189e-06 9.04378131e-06 7.95019134e-06 9.79795159e-06
 9.02499366e-06 9.06648713e-06 9.17535904e-06 9.05913923e-06
 9.08575939e-06 9.04592476e-06 9.56968418e-06 8.42950866e-06
 8.38011953e-06 9.84160269e-06 1.04967931e-05 9.09522598e-06
 8.93833842e-06 8.43223198e-06 9.12710195e-06 9.08812949e-06
 7.94100773e-06 8.59870839e-06 9.07304275e-06 9.70486016e-06
 7.78462208e-06 8.57340255e-06 9.02583928e-06 7.89146370e-06
 9.06803087e-06 9.68646993e-06 9.22318060e-06 9.08331367e-06
 9.09837908e-06 9.12435780e-06 6.87133947e-06 8.45743630e-06
 9.74059079e-06 8.42456225e-06 7.82022572e-06 9.23043664e-06
 9.60127204e-06 9.80275229e-06 9.78272239e-06 9.73562531e-06
 9.08906672e-06 9.10413150e-06 8.03968702e-06]
-0.0018322682860596024


In [11]:
print(len(obs_s))

2213


In [12]:
#Convert WRF file time into same units as the obs_seq time
dt_tot = (dt - datetime(1601,1,1)).total_seconds() / 86400

time_diff = np.abs(dt_tot - time_T2)

print(time_diff[time_diff<(150/86400)])

print(dt_tot)
print(time_diff)

[0.0011689814855344594 0.0010879629699047655 0.0010648148017935455
 0.0010532407322898507 0.0009027777705341578 0.0008680555620230734
 0.00074074073927477 0.0006481481541413814 0.0006018518470227718
 0.00035879630013369024 0.00023148147738538682 2.3148139007389545e-05
 1.1574069503694773e-05 1.1574069503694773e-05 2.3148139007389545e-05
 2.3148139007389545e-05 3.472220851108432e-05 3.472220851108432e-05
 3.472220851108432e-05 4.629630711860955e-05 4.629630711860955e-05
 4.629630711860955e-05 5.787037662230432e-05 6.944444612599909e-05
 8.101851562969387e-05 8.101851562969387e-05 0.00011574075324460864
 0.00015046296175569296 0.00016203703125938773 0.0001736111007630825
 0.0001967592688743025 0.00023148147738538682 0.0003124999930150807
 0.00040509257814846933 0.00046296295477077365 0.00046296295477077365
 0.00048611112288199365 0.0005092592618893832 0.0005439814704004675
 0.0005671296385116875 0.0005671296385116875 0.0006712962931487709
 0.0007060185307636857 0.00074074073927477 0.0007

In [13]:
print(dt.isoformat()[14:16])

00


In [14]:
print(len(obs_s))

2213
